# AniList Anime Dataset — Data Cleaning Script
> Nova IMS · 2025/2026 · Group 16

In [106]:
"""
Steps:
  1. Load CSV
  2. Drop useless columns
  3. Fix data types
  4. Basic data quality (nulls, duplicates)
  5. Parse JSON columns → Python objects
  6. Extract separate collections (characters, staff, studios, tags, reviews)
  7. Save 6 JSON files → 6 MongoDB collections
"""

import json
import pandas as pd
import os

---
# 1. Load

In [107]:
INPUT_PATH = "../Data/anilist_anime_data_complete.csv"

print("Loading CSV...")
df = pd.read_csv(INPUT_PATH)
print(f"  Loaded: {len(df)} rows × {len(df.columns)} columns")

Loading CSV...
  Loaded: 20329 rows × 62 columns


---
# 2. Drop Useless columns
### What exactly dropped:

**'Unnamed: 0'** - pandas index<br>
**'type'**       - always "ANIME", no value<br>
**'isFavourite'** 'isLocked' - user-session flags, irrelevant for DB<br>
**'hashtag', 'chapters', 'volumes'** - near 100% null coverImage_large / _medium - redundant<br>
**title_userPreferred** - redundant with title_romaji<br>

In [108]:
DROP_COLS = [
    # pandas / dataset artifacts
    "Unnamed: 0", "type", "isFavourite", "isLocked",
    # manga-only fields
    "hashtag", "chapters", "volumes",
    # redundant image & title variants
    "coverImage_large", "coverImage_medium", "title_userPreferred",
    # redundant with coverImage
    "bannerImage",
    # redundant with averageScore
    "meanScore",
    # alternative titles — 36% null, low priority
    "synonyms",
    # external links — not our platform
    "siteUrl", "externalLinks",
    # low-value metadata
    "seasonInt", "updatedAt", "idMal",
    # AniList-specific editorial — not our platform's ranking
    "rankings",
    # derivable from the DB itself
    "recommendations",
    # near-empty columns (86.2% / 99.9% / 76.1% null)
    "streamingEpisodes", "nextAiringEpisode", "airingSchedule",
]
df.drop(columns=DROP_COLS, inplace=True, errors="ignore")
print(f"  After drop: {len(df.columns)} columns")

  After drop: 39 columns


---
# 3. Fixing Data Types


In [109]:
#Integers
int_cols = [
    "idMal", "startDate_year", "startDate_month", "startDate_day",
    "endDate_year", "endDate_month", "endDate_day",
    "seasonYear", "seasonInt", "episodes", "duration",
    "averageScore", "meanScore", "popularity", "favourites", "trending",
]
for col in int_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

In [110]:
#Bool
bool_cols = ["isLicensed", "isAdult"]
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].map({True: True, False: False, "True": True, "False": False})

In [111]:
#TimeStamps to datetime
if "updatedAt" in df.columns:
    df["updatedAt"] = pd.to_datetime(df["updatedAt"], unit="s", errors="coerce")

In [112]:
def build_date(row, prefix):
    y = row.get(f"{prefix}_year")
    if pd.isna(y):
        return None
    m = row.get(f"{prefix}_month")
    d = row.get(f"{prefix}_day")
    return {
        "year":  int(y),
        "month": None if pd.isna(m) else int(m),
        "day":   None if pd.isna(d) else int(d),
    }

df["startDate"] = df.apply(lambda r: build_date(r, "startDate"), axis=1)
df["endDate"]   = df.apply(lambda r: build_date(r, "endDate"),   axis=1)

df.drop(columns=[
    "startDate_year","startDate_month","startDate_day",
    "endDate_year","endDate_month","endDate_day",
], inplace=True, errors="ignore")

In [113]:
#Titles
df["title"] = df.apply(lambda r: {
    "romaji":  r.get("title_romaji"),
    "english": r.get("title_english"),
    "native":  r.get("title_native"),
}, axis=1)
df.drop(columns=["title_romaji","title_english","title_native"], inplace=True, errors="ignore")

In [114]:
#CoverImages
df["coverImage"] = df.apply(lambda r: {
    "extraLarge": r.get("coverImage_extraLarge"),
    "color":      r.get("coverImage_color"),
}, axis=1)
df.drop(columns=["coverImage_extraLarge","coverImage_color"], inplace=True, errors="ignore")

In [115]:
#Trailers
df["trailer"] = df.apply(lambda r: {
    "id":        r.get("trailer_id"),
    "site":      r.get("trailer_site"),
    "thumbnail": r.get("trailer_thumbnail"),
} if pd.notna(r.get("trailer_id")) else None, axis=1)
df.drop(columns=["trailer_id","trailer_site","trailer_thumbnail"], inplace=True, errors="ignore")

In [116]:
print("  Data types fixed, nested fields reconstructed")

  Data types fixed, nested fields reconstructed


---
# 4. Basic Data Quality


In [117]:
before = len(df)
df.drop_duplicates(subset=["id"], inplace=True)
print(f"  Duplicates removed: {before - len(df)} rows")

# Drop rows with no meaningful identity (no id or no title)
df = df[df["id"].notna()]
print(f"  Rows after null-id drop: {len(df)}")

for score_col in ["averageScore", "meanScore"]:
    if score_col in df.columns:
        invalid = ((df[score_col] < 0) | (df[score_col] > 100)).sum()
        if invalid > 0:
            print(f"  WARNING: {invalid} out-of-range values in '{score_col}' — clamping")
            df[score_col] = df[score_col].clip(0, 100)

  Duplicates removed: 0 rows
  Rows after null-id drop: 20329


---
# 5. Parsing JSON Columns

In [118]:
JSON_COLS = [
    "genres", "synonyms", "tags", "rankings", "externalLinks",
    "streamingEpisodes", "relations", "characters", "staff", "studios",
    "airingSchedule", "recommendations", "reviews",
    "stats_scoreDistribution", "stats_statusDistribution",
    "nextAiringEpisode",
]

def safe_parse(val):
    if isinstance(val, (list, dict)):
        return val
    if not isinstance(val, str):
        return None
    if val in ("", "[]", "{}"):
        return None
    try:
        return json.loads(val)
    except (json.JSONDecodeError, TypeError):
        return None

for col in JSON_COLS:
    if col in df.columns:
        df[col] = df[col].apply(safe_parse)

print("  JSON columns parsed")

  JSON columns parsed


---
# 6. Final Reports

In [119]:
print("\n=== Final shape ===")
print(f"Rows   : {len(df)}")
print(f"Columns: {len(df.columns)}")
print(f"Columns: {list(df.columns)}")

print("\n=== Null % per column ===")
nulls = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(nulls[nulls > 0].round(1).to_string())


=== Final shape ===
Rows   : 20329
Columns: 30
Columns: ['id', 'format', 'status', 'description', 'season', 'seasonYear', 'episodes', 'duration', 'countryOfOrigin', 'isLicensed', 'source', 'genres', 'tags', 'averageScore', 'popularity', 'favourites', 'trending', 'isAdult', 'relations', 'characters', 'staff', 'studios', 'reviews', 'stats_scoreDistribution', 'stats_statusDistribution', 'startDate', 'endDate', 'title', 'coverImage', 'trailer']

=== Null % per column ===
reviews         82.3
trailer         63.1
characters      36.1
season          31.3
seasonYear      31.3
relations       30.4
averageScore    20.6
tags            19.2
studios         18.6
staff           13.7
genres          13.1
source          11.4
description      6.1
endDate          1.2
duration         0.9
episodes         0.8
format           0.0


In [120]:
import subprocess

result = subprocess.run(
    ["mongosh", "--quiet", "--eval",
     "db.getCollectionNames().forEach(c => print(c))",
     "anilist_db"],
    capture_output=True, text=True
)

if result.returncode == 0:
    cols = [c for c in result.stdout.strip().splitlines() if c]
    print(f"Collections in 'anilist_db' ({len(cols)} total):")
    for c in cols:
        print(f"  • {c}")
else:
    print("mongosh error:", result.stderr)

Collections in 'anilist_db' (6 total):
  • characters
  • staff
  • tags
  • studios
  • reviews
  • anime


---
# 7. Extract Collections


In [121]:
SEPARATE_COLS = ["characters", "studios", "tags", "reviews"]

def iter_items(val):
    if isinstance(val, list):
        return val
    if isinstance(val, dict):
        return val.get("nodes") or val.get("edges") or []
    return []

def unwrap_node(item):
    """Flatten GraphQL edge pattern: {isMain, node: {id, name, ...}} → {id, name, ..., isMain}"""
    if not isinstance(item, dict) or "node" not in item:
        return item
    node = item["node"]
    if not isinstance(node, dict):
        return item
    return {**node, **{k: v for k, v in item.items() if k != "node"}}

extracted = {}

for col in SEPARATE_COLS:
    seen, docs, id_lists = set(), [], []

    for val in df[col]:
        items = iter_items(val) if isinstance(val, (list, dict)) else []
        row_ids = []
        for item in items:
            item = unwrap_node(item)
            if not isinstance(item, dict):
                continue
            uid = item.get("id")
            if uid is not None:
                row_ids.append(uid)
                if uid not in seen:
                    seen.add(uid)
                    docs.append(item)
            else:
                docs.append(item)
        id_lists.append(row_ids if row_ids else None)

    extracted[col] = docs
    df[col] = id_lists
    print(f"  {col:15s}: {len(docs):6d} unique docs extracted")

print(f"\n  anime: {len(df)} documents (nested arrays replaced with ID refs)")

  characters     : 144294 unique docs extracted
  studios        :  39743 unique docs extracted
  tags           :    418 unique docs extracted
  reviews        :  13611 unique docs extracted

  anime: 20329 documents (nested arrays replaced with ID refs)


---
# 8. Saving it as JSON
> 6 files → 6 collections in MongoDB

In [122]:
import numpy as np

def convert(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, pd.Timestamp):
        return obj.isoformat()
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

def sanitize(records):
    return [
        {k: (None if isinstance(v, (float, np.floating)) and not np.isfinite(v) else v)
         for k, v in r.items()}
        for r in records
    ]

# anime collection
anime_records = sanitize(df.where(pd.notna(df), None).to_dict(orient="records"))
anime_path = "../Data/anime.json"
with open(anime_path, "w", encoding="utf-8") as f:
    json.dump(anime_records, f, ensure_ascii=False, default=convert)
print(f"  anime          : {len(anime_records):6d} docs → {anime_path}")

# separate collections
for col, docs in extracted.items():
    out_path = f"../Data/{col}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(docs, f, ensure_ascii=False, default=convert)
    print(f"  {col:15s}: {len(docs):6d} docs → {out_path}")

print("\n--- mongoimport commands ---")
for col in ["anime"] + list(extracted.keys()):
    print(f"mongoimport --db anilist_db --collection {col} --file Data/{col}.json --jsonArray")

  anime          :  20329 docs → ../Data/anime.json
  characters     : 144294 docs → ../Data/characters.json
  studios        :  39743 docs → ../Data/studios.json
  tags           :    418 docs → ../Data/tags.json
  reviews        :  13611 docs → ../Data/reviews.json

--- mongoimport commands ---
mongoimport --db anilist_db --collection anime --file Data/anime.json --jsonArray
mongoimport --db anilist_db --collection characters --file Data/characters.json --jsonArray
mongoimport --db anilist_db --collection studios --file Data/studios.json --jsonArray
mongoimport --db anilist_db --collection tags --file Data/tags.json --jsonArray
mongoimport --db anilist_db --collection reviews --file Data/reviews.json --jsonArray


In [123]:
MONGO_HOST = "mongodb://localhost:27017"
MONGO_DB   = "anilist_db"

collections = ["anime"] + list(extracted.keys())

print("Importing into MongoDB...")
for col in collections:
    result = subprocess.run([
        "mongoimport",
        "--uri",        MONGO_HOST,
        "--db",         MONGO_DB,
        "--collection", col,
        "--file",       f"../Data/{col}.json",
        "--jsonArray",
        "--drop",
    ], capture_output=True, text=True)

    status = "OK" if result.returncode == 0 else "FAILED"
    print(f"  [{status}] {col:15s} — {result.stderr.strip().splitlines()[-1] if result.stderr else ''}")

print("\nDone. Connect: mongodb://localhost:27017")

Importing into MongoDB...
  [FAILED] anime           — 2026-05-13T16:10:03.843+0100	1 document(s) imported successfully. 0 document(s) failed to import.
  [OK] characters      — 2026-05-13T16:10:10.401+0100	144294 document(s) imported successfully. 0 document(s) failed to import.
  [OK] studios         — 2026-05-13T16:10:11.182+0100	39743 document(s) imported successfully. 0 document(s) failed to import.
  [OK] tags            — 2026-05-13T16:10:11.219+0100	418 document(s) imported successfully. 0 document(s) failed to import.
  [OK] reviews         — 2026-05-13T16:10:11.534+0100	13611 document(s) imported successfully. 0 document(s) failed to import.

Done. Connect: mongodb://localhost:27017


In [124]:
all_collections = {"anime": df, **{col: pd.DataFrame(docs) for col, docs in extracted.items()}}

for name, data in all_collections.items():
    frame = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
    print(f"\n{'='*50}")
    print(f"  {name.upper()}")
    print(f"{'='*50}")
    print(f"  Rows   : {len(frame)}")
    print(f"  Columns: {len(frame.columns)}")
    print(f"  Fields : {list(frame.columns)}")

    nulls = (frame.isnull().sum() / len(frame) * 100).sort_values(ascending=False)
    non_zero = nulls[nulls > 0].round(1)
    if non_zero.empty:
        print("  Null % : none")
    else:
        print("  Null % per column:")
        print(non_zero.to_string())


  ANIME
  Rows   : 20329
  Columns: 30
  Fields : ['id', 'format', 'status', 'description', 'season', 'seasonYear', 'episodes', 'duration', 'countryOfOrigin', 'isLicensed', 'source', 'genres', 'tags', 'averageScore', 'popularity', 'favourites', 'trending', 'isAdult', 'relations', 'characters', 'staff', 'studios', 'reviews', 'stats_scoreDistribution', 'stats_statusDistribution', 'startDate', 'endDate', 'title', 'coverImage', 'trailer']
  Null % per column:
reviews         82.3
trailer         63.1
characters      36.1
season          31.3
seasonYear      31.3
relations       30.4
averageScore    20.6
tags            19.2
studios         18.6
staff           13.7
genres          13.1
source          11.4
description      6.1
endDate          1.2
duration         0.9
episodes         0.8
format           0.0

  CHARACTERS
  Rows   : 144294
  Columns: 6
  Fields : ['id', 'name', 'image', 'description', 'role', 'voiceActors']
  Null % per column:
name           99.9
description    30.1

  

---
# 9. Collection Samples
> First 10 documents per collection — for visual orientation.

In [125]:
from IPython.display import display

all_collections = {"anime": df, **{col: pd.DataFrame(docs) for col, docs in extracted.items()}}

for name, data in all_collections.items():
    frame = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
    print(f"\n{'='*60}")
    print(f"  {name.upper()} — {len(frame)} total docs")
    print(f"{'='*60}")
    display(frame.head(10))


  ANIME — 20329 total docs


,id,format,status,description,season,seasonYear,episodes,duration,countryOfOrigin,isLicensed,...,staff,studios,reviews,stats_scoreDistribution,stats_statusDistribution,startDate,endDate,title,coverImage,trailer
0,1497,MOVIE,FINISHED,The scene is set with a poster on a street cor...,FALL,1962,1,38,JP,True,...,"[{'id': 325739, 'role': 'Script Composition', ...",[3736],None,"[{'score': 10, 'amount': 13}, {'score': 20, 'a...","[{'status': 'CURRENT', 'amount': 24}, {'status...","{'year': 1962, 'month': 11, 'day': 5}","{'year': 1962, 'month': 11, 'day': 5}","{'romaji': 'Aru Machi Kado no Monogatari', 'en...",{'extraLarge': 'https://s4.anilist.co/file/ani...,None
1,1547,TV,FINISHED,"Q-taro, a monster, is living with the Ohara fa...",SUMMER,1965,96,25,JP,True,...,"[{'id': 5749, 'role': 'Original Creator', 'nod...","[3849, 27201]",None,"[{'score': 10, 'amount': 14}, {'score': 20, 'a...","[{'status': 'CURRENT', 'amount': 21}, {'status...","{'year': 1965, 'month': 8, 'day': 29}","{'year': 1967, 'month': 6, 'day': 28}","{'romaji': 'Obake no Q-tarou', 'english': nan,...",{'extraLarge': 'https://s4.anilist.co/file/ani...,None
2,1572,TV,FINISHED,"The series follows the adventures Leo, a young...",FALL,1965,52,23,JP,True,...,"[{'id': 724627, 'role': 'ADR Director (Brazili...","[3921, 3922, 3923]",[27621],"[{'score': 10, 'amount': 11}, {'score': 20, 'a...","[{'status': 'CURRENT', 'amount': 110}, {'statu...","{'year': 1965, 'month': 10, 'day': 6}","{'year': 1966, 'month': 9, 'day': 28}","{'romaji': 'Jungle Taitei', 'english': 'Kimba ...",{'extraLarge': 'https://s4.anilist.co/file/ani...,None
3,1982,MOVIE,FINISHED,A cat just wants some alone time with his girl...,FALL,1962,1,3,JP,True,...,"[{'id': 3572, 'role': 'Director', 'node': {'id...",[4672],None,"[{'score': 10, 'amount': 71}, {'score': 20, 'a...","[{'status': 'CURRENT', 'amount': 9}, {'status'...","{'year': 1962, 'month': 11, 'day': 5}","{'year': 1962, 'month': 11, 'day': 5}","{'romaji': 'Osu', 'english': 'Male', 'native':...",{'extraLarge': 'https://s4.anilist.co/file/ani...,"{'id': 'MyN1NXlfJG0', 'site': 'youtube', 'thum..."
4,1984,MOVIE,FINISHED,"This is a short, privately produced animated f...",FALL,1964,1,5,JP,True,...,"[{'id': 3562, 'role': 'Director', 'node': {'id...",[4675],None,"[{'score': 10, 'amount': 34}, {'score': 20, 'a...","[{'status': 'CURRENT', 'amount': 10}, {'status...","{'year': 1964, 'month': 9, 'day': 21}","{'year': 1964, 'month': 9, 'day': 21}","{'romaji': 'Memory', 'english': nan, 'native':...",{'extraLarge': 'https://s4.anilist.co/file/ani...,None
5,1989,MOVIE,FINISHED,A man trapped on a raft in the middle of the o...,FALL,1965,1,4,JP,True,...,"[{'id': 3581, 'role': 'Director', 'node': {'id...",[4679],None,"[{'score': 10, 'amount': 32}, {'score': 20, 'a...","[{'status': 'CURRENT', 'amount': 11}, {'status...","{'year': 1965, 'month': 10, 'day': 1}","{'year': 1965, 'month': 10, 'day': 1}","{'romaji': 'Shizuku', 'english': 'The Drop', '...",{'extraLarge': 'https://s4.anilist.co/file/ani...,None
6,2047,MOVIE,FINISHED,In a fictional place where using the imaginati...,FALL,1964,1,8,JP,True,...,"[{'id': 3569, 'role': 'Director', 'node': {'id...",[4778],None,"[{'score': 10, 'amount': 22}, {'score': 20, 'a...","[{'status': 'CURRENT', 'amount': 9}, {'status'...","{'year': 1964, 'month': 9, 'day': 21}","{'year': 1964, 'month': 9, 'day': 21}","{'romaji': 'Ningyo', 'english': 'Mermaid', 'na...",{'extraLarge': 'https://s4.anilist.co/file/ani...,None
7,2686,TV,FINISHED,Dr.Haneda was developing experimental giant ro...,FALL,1963,83,25,JP,True,...,"[{'id': 914377, 'role': 'ADR Director (English...","[9260, 26030, 26031, 26780]",None,"[{'score': 10, 'amount': 15}, {'score': 20, 'a...","[{'status': 'CURRENT', 'amount': 105}, {'statu...","{'year': 1963, 'month': 10, 'day': 20}","{'year': 1965, 'month': 5, 'day': 27}","{'romaji': 'Tetsujin 28-gou', 'english': 'Giga...",{'extraLarge': 'https://s4.anilist.co/file/ani...,None
8,2747,TV,FINISHED,"In the distant year 2003, Japan is a techn


  CHARACTERS — 144294 total docs


,id,name,image,description,role,voiceActors
0,450688,NaN,{'large': 'https://s4.anilist.co/file/anilistc...,Hyoutantsugi is a gag character that frequentl...,BACKGROUND,[]
1,498476,NaN,{'large': 'https://s4.anilist.co/file/anilistc...,The city thug moth is one of the main characte...,MAIN,[]
2,498477,NaN,{'large': 'https://s4.anilist.co/file/anilistc...,The mischievous mouse Kanku is one of the main...,MAIN,[]
3,359338,NaN,{'large': 'https://s4.anilist.co/file/anilistc...,NaN,MAIN,"[{'id': 107793, 'name': {'full': 'Machiko Soga..."
4,61911,NaN,{'large': 'https://s4.anilist.co/file/anilistc...,The main character of the story. He believes ...,MAIN,"[{'id': 96846, 'name': {'full': 'Cristina Hern..."
5,308338,NaN,{'large': 'https://s4.anilist.co/file/anilistc...,NaN,SUPPORTING,"[{'id': 175518, 'name': {'full': 'Bruno Carnev..."
6,217202,NaN,{'large': 'https://s4.anilist.co/file/anilistc...,NaN,MAIN,[]
7,217168,NaN,{'large': 'https://s4.anilist.co/file/anilistc...,NaN,MAIN,[]
8,217169,NaN,{'large': 'https://s4.anilist.co/file/anilistc...,NaN,MAIN,[]
9,405942,NaN,{'large': 'https://s4.anilist.co/file/anilistc...,He has a rather special power and in some ways...,SUPPORTING,"[{'id': 226708, 'name': {'full': 'Shirou Kuno'..."



  STUDIOS — 39743 total docs


,id,name,isAnimationStudio,isMain
0,3736,Mushi Production,True,False
1,3849,Shin-Ei Animation,True,False
2,27201,Tokyo Movie Shinsha,True,False
3,3921,Mushi Production,True,True
4,3922,The Right Stuf International,False,False
5,3923,Media Blasters,False,False
6,4672,Mushi Production,True,False
7,4675,Mushi Production,True,False
8,4679,Mushi Production,True,False
9,4778,Mushi Production,True,False



  TAGS — 418 total docs


,id,name,description,category,rank,isGeneralSpoiler,isMediaSpoiler,isAdult
0,341,No Dialogue,This work contains no dialogue.,Technical,76,False,False,False
1,111,War,Partly or completely set during wartime.,Theme-Other,70,False,True,False
2,595,Urban,Partly or completely set in a city.,Setting-Scene,66,False,False,False
3,85,Tragedy,Centers around tragic events and unhappy endings.,Theme-Drama,60,True,True,False
4,1672,Jazz Music,"Centers on the musical style of jazz, not to b...",Theme-Arts-Music,60,False,False,False
5,25,Historical,Partly or completely set during a real period ...,Setting-Time,53,False,False,False
6,103,Politics,"Centers around politics, politicians, or gover...",Theme-Other,40,False,True,False
7,433,Animals,Prominently features animal characters in a le...,Theme-Other,40,False,False,False
8,46,School,Partly or completely set in a primary or secon...,Setting-Scene,50,False,False,False
9,710,Achromatic,Contains animation that is primarily done in b...,Technical,20,False,False,False



  REVIEWS — 13611 total docs


,id,summary,rating,score
0,27621,"A repetitive, sloppy, and intellectually insul...",4,50
1,13494,I've watched an anime from 1960. Have you?,16,60
2,26719,Animation as Smooth as a Snake for the Snake Y...,4,60
3,17748,"Small review of this short, recommended that y...",14,20
4,7093,very very cool moenkey swings in his swing,13,100
5,2833,Ashita no Joe is a true classic,19,85
6,7079,A review of Joe that I made years ago,18,95
7,7370,A classic in every way that still holds up today.,53,95
8,9353,Joe is a iconic series. The Drama in this Anim...,31,90
9,14432,Still popular in Japan after 50 years and I ca...,36,100
